# Risque de marché : VaR, GARCH et backtesting
Ce notebook appelle `src/market_risk.py`. Les chiffres sont recalculés à l'exécution.

In [1]:
import sys; sys.path.insert(0,'..')
import pandas as pd, numpy as np
from src import market_risk as mr
R=pd.read_csv('../data/log_returns.csv',index_col=0,parse_dates=True)
P=mr.portfolio_returns(R)
print(len(P), P.index[0].date(), P.index[-1].date())
print('vol annualisée', round(P.std()*252**.5,4), '| asymétrie', round(P.skew(),2), '| excès de kurtosis', round(P.kurt(),1))

1434 2019-01-03 2024-12-30
vol annualisée 0.1626 | asymétrie -1.08 | excès de kurtosis 16.1


## VaR et CVaR (en % du portefeuille)

In [2]:
rows={}
for a in (.95,.99):
    rows[f'{a:.0%}']={'historique':mr.var_historical(P,a),'normale':mr.var_normal(P,a),'student-t':mr.var_student(P,a),'MC normal':mr.var_monte_carlo_normal(P,a),'MC student-t':mr.var_monte_carlo_t(P,a),'CVaR hist.':mr.cvar_historical(P,a)}
(pd.DataFrame(rows)*100).round(3)

,95%,99%
historique,1.450,2.877
normale,1.648,2.346
student-t,1.349,2.697
MC normal,1.654,2.347
MC student-t,1.354,2.690
CVaR hist.,2.531,4.548


## GARCH(1,1) skew-t
Le quantile doit être celui de la loi estimée. Comparaison des deux choix à 99 %.

In [3]:
res=mr.fit_garch(P)
print(res.params.round(4).to_string())
for inn in ('skewt','normal'):
    v=mr.violations(P,mr.var_garch(res,.99,inn))
    print(inn, mr.kupiec_pof(v,.99), '\n   ', mr.christoffersen_ind(v))

mu          0.0462
omega       0.0182
alpha[1]    0.0918
beta[1]     0.8881
eta         5.5493
lambda     -0.1820
skewt {'T': 1434, 'violations': 18, 'expected': 14.3, 'rate': 0.0126, 'LR': 0.873, 'p_value': 0.3501, 'reject_5pct': False} 
    {'n00': 1397, 'n01': 18, 'n10': 18, 'n11': 0, 'LR': 0.458, 'p_value': 0.4986, 'reject_5pct': False}
normal {'T': 1434, 'violations': 34, 'expected': 14.3, 'rate': 0.0237, 'LR': 19.658, 'p_value': 0.0, 'reject_5pct': True} 
    {'n00': 1367, 'n01': 32, 'n10': 32, 'n11': 2, 'LR': 1.333, 'p_value': 0.2483, 'reject_5pct': False}


## VaR historique glissante hors échantillon (250 jours)

In [4]:
roll=mr.rolling_hist_var_violations(P,.99,250)
print(mr.kupiec_pof(roll,.99)); print(mr.christoffersen_ind(roll))
k=int(roll[-250:].sum()); print('250 derniers jours :',k,'violations -> zone',mr.basel_zone(k))

{'T': 1184, 'violations': 22, 'expected': 11.8, 'rate': 0.0186, 'LR': 7.029, 'p_value': 0.008, 'reject_5pct': True}
{'n00': 1143, 'n01': 18, 'n10': 18, 'n11': 4, 'LR': 12.336, 'p_value': 0.0004, 'reject_5pct': True}
250 derniers jours : 4 violations -> zone verte
